In [314]:
import sys
print(sys.executable)

c:\ML\sentiment_analysis\venv\Scripts\python.exe


# Financial Sentiment Classifier
Classifies financial news sentences as positive, negative, or neutral.

**Dataset**: Financial PhraseBank  
**Model**: TF-IDF + Logistic Regression  
**Accuracy**: 0.77 | **Macro F1**: 0.73

In [ ]:
# importing pandas and reading the dataset
import pandas as pd

df = pd.read_csv("all-data.csv", header=None,encoding = 'latin1')
df.head()

,0,1
0,neutral,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing ."
1,neutral,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said ."
2,negative,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported ."
3,positive,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
4,positive,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales ."


In [ ]:
# naming the columns
df.columns = ['Label','Sentence']

In [317]:
df.head()

,Label,Sentence
0,neutral,"According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing ."
1,neutral,"Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said ."
2,negative,"The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported ."
3,positive,With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
4,positive,"According to the company 's updated strategy for the years 2009-2012 , Basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales ."


In [ ]:
# checking number of statements of each type
print(df['Label'].value_counts())

Label
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64


In [ ]:
# lowercasing the strings
df['Sentence'] = df['Sentence'].str.lower()

In [320]:
df.head()

,Label,Sentence
0,neutral,"according to gran , the company has no plans to move all production to russia , although that is where the company is growing ."
1,neutral,"technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said ."
2,negative,"the international electronic industry company elcoteq has laid off tens of employees from its tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily postimees reported ."
3,positive,with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .
4,positive,"according to the company 's updated strategy for the years 2009-2012 , basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales ."


In [ ]:
# importing nltk to tokenize the dataset
import nltk
from nltk.tokenize import word_tokenize
df['tokens'] = df['Sentence'].apply(word_tokenize)

In [322]:
df.head()

,Label,Sentence,tokens
0,neutral,"according to gran , the company has no plans to move all production to russia , although that is where the company is growing .","[according, to, gran, ,, the, company, has, no, plans, to, move, all, production, to, russia, ,, although, that, is, where, the, company, is, growing, .]"
1,neutral,"technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .","[technopolis, plans, to, develop, in, stages, an, area, of, no, less, than, 100,000, square, meters, in, order, to, host, companies, working, in, computer, technologies, and, telecommunications, ,, the, statement, said, .]"
2,negative,"the international electronic industry company elcoteq has laid off tens of employees from its tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily postimees reported .","[the, international, electronic, industry, company, elcoteq, has, laid, off, tens, of, employees, from, its, tallinn, facility, ;, contrary, to, earlier, layoffs, the, company, contracted, the, ranks, of, its, office, workers, ,, the, daily, postimees, reported, .]"
3,positive,with the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the production profitability .,"[with, the, new, production, plant, the, company, would, increase, its, capacity, to, meet, the, expected, increase, in, demand, and, would, improve, the, use, of, raw, materials, and, therefore, increase, the, production, profitability, .]"
4,positive,"according to the company 's updated strategy for the years 2009-2012 , basware targets a long-term net sales growth in the range of 20 % -40 % with an operating profit margin of 10 % -20 % of net sales .","[according, to, the, company, 's, updated, strategy, for, the, years, 2009-2012, ,, basware, targets, a, long-term, net, sales, growth, in, the, range, of, 20, %, -40, %, with, an, operating, profit, margin, of, 10, %, -20, %, of, net, sales, .]"


In [ ]:
# removing punctuation but keeping some that would have relevant meaning 
keep_punc = ['%','$','?']
import string
df['tokens'] = df['tokens'].apply(lambda tokens:[word for word in tokens if word not in string.punctuation or word in keep_punc])


In [ ]:
# listing stopwords
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
print(list(stop_words))

['him', 'which', 'above', 'not', 'who', 'same', "shouldn't", "isn't", "you've", 'couldn', 'below', 'does', 'only', 'other', 'being', 'very', "i'll", 'isn', 'from', 'will', 'yourself', "i'd", "we'll", 'as', 'shouldn', "they'd", 'it', 'what', "mightn't", 'now', 'won', 'whom', 'can', 'most', 've', 'these', 'yours', "don't", 'of', 'an', 'to', 'herself', "aren't", "hasn't", 'under', 's', 'you', 'himself', 'ain', 'didn', 'their', "she'll", 'this', 'the', 'a', 'own', 'each', "should've", 'my', 'doing', 'and', 'if', 'did', 'had', 'yourselves', 'hasn', 'o', 're', "didn't", 'all', 'ma', 'so', 'i', "he'd", 'theirs', 'myself', 'while', 'its', 'mustn', "won't", "wouldn't", 'few', 'when', "mustn't", 'your', 'because', 'haven', 'me', 'hadn', 'should', 'weren', 'm', "they're", "she'd", 'were', 'both', 'or', 'on', 'those', "we'd", "he'll", "i've", "they'll", 'too', 'until', 'during', 'hers', 'there', "we're", 'once', 'by', 'off', "it'll", 'they', "weren't", "they've", 'wasn', "couldn't", 'again', 'do',

In [ ]:
# counting the frequency of each stopword in the dataset
from collections import Counter
stopwords_counts = Counter()

for tokens in df['tokens']:
    for word in tokens:
        if word in stop_words:
            stopwords_counts[word] += 1


In [327]:
print(stopwords_counts)

Counter({'the': 6066, 'of': 3213, 'in': 2969, 'and': 2594, 'to': 2509, 'a': 1715, 'for': 1152, 'is': 930, 'will': 850, 'from': 769, 'on': 681, 'its': 646, 'has': 578, 'with': 573, 'by': 558, 'as': 542, 'be': 542, 'at': 481, 'it': 472, 'that': 434, 'was': 370, 'an': 326, 'are': 286, 'm': 281, 'which': 221, 'have': 219, 'this': 193, 'up': 182, 'been': 167, 'about': 153, 'we': 124, 'were': 123, 'other': 118, 'our': 115, 'than': 114, 'more': 103, 'their': 103, 'or': 101, 'not': 99, 'all': 96, 'down': 95, 'some': 94, 'over': 87, 'after': 84, 'into': 79, 'same': 78, 'had': 78, 'can': 77, 'under': 72, 'while': 71, 'both': 68, 'now': 66, 'most': 59, 'through': 59, 'between': 58, 'before': 58, 'during': 57, 'these': 56, 'he': 56, 'no': 53, 'they': 53, 'further': 52, 'out': 51, 's': 47, 'when': 45, 'such': 44, 'his': 38, 'off': 35, 'won': 34, 'where': 32, 'own': 32, 'there': 30, 'being': 29, 'very': 28, 'who': 27, 'only': 26, 'any': 26, 'i': 24, 'them': 24, 'but': 22, 'if': 22, 'because': 21, 'b

In [ ]:
# keeping stopwords that have relevant meaning 
keep_stopword = ['up','not','down','over','after','no','against','below','won','off']
def remove_stopwords(tokens):
   return [word for word in tokens if word not in stop_words or word in keep_stopword] 

In [329]:
df['tokens'] = df['tokens'].apply(remove_stopwords)

In [ ]:
# applying 'isnumeric' to the tokens
df['tokens'] = df['tokens'].apply(
    lambda tokens: [word for word in tokens if not word.isnumeric()]
)

In [ ]:
# joining the tokens into a sentence to apply tfidf
from sklearn.feature_extraction.text import TfidfVectorizer
df['clean_text'] = df['tokens'].apply(lambda x: ' '.join(x))


In [ ]:
# applying TF-IDF 
v = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
transformed_output = v.fit_transform(df['clean_text'])
print(v.vocabulary_)

all_feature_names = v.get_feature_names_out()
for word in all_feature_names:
    indx = v.vocabulary_.get(word)
    print(f"{word}{v.idf_[indx]}")


{'according': np.int64(533), 'gran': np.int64(4057), 'company': np.int64(1948), 'no': np.int64(6106), 'plans': np.int64(6896), 'move': np.int64(5914), 'production': np.int64(7100), 'russia': np.int64(7818), 'although': np.int64(790), 'growing': np.int64(4141), 'technopolis': np.int64(9096), 'develop': np.int64(2482), 'stages': np.int64(8639), 'area': np.int64(953), 'less': np.int64(5131), '100': np.int64(44), '000': np.int64(3), 'square': np.int64(8617), 'meters': np.int64(5678), 'order': np.int64(6440), 'host': np.int64(4387), 'companies': np.int64(1936), 'working': np.int64(9927), 'computer': np.int64(2086), 'technologies': np.int64(9074), 'telecommunications': np.int64(9121), 'statement': np.int64(8679), 'said': np.int64(7851), 'technopolis plans': np.int64(9097), '100 000': np.int64(45), '000 square': np.int64(24), 'square meters': np.int64(8620), 'statement said': np.int64(8681), 'international': np.int64(4702), 'electronic': np.int64(2795), 'industry': np.int64(4602), 'elcoteq': 

In [ ]:
# Create feature matrix and target labels for training
X = v.fit_transform(df['clean_text'])
y = df['Label']

In [ ]:
# Splitting the data into train and test
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Training the model on the train data
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(class_weight='balanced',max_iter=1000)
model.fit(X_train, y_train) 

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

In [ ]:
# Predicting the model
y_pred = model.predict(X_test)

In [ ]:
# Printing the classification report 
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.70      0.69      0.70       110
     neutral       0.80      0.87      0.83       571
    positive       0.72      0.62      0.67       289

    accuracy                           0.77       970
   macro avg       0.74      0.73      0.73       970
weighted avg       0.77      0.77      0.77       970

